# Training Example Visualization

Single qualitative figure that shows what **one supervision example** looks like during training — the picture readers usually want before they see Hits@10 numbers.

Layout, by design:

- **Right side**: paper $u$ from the supervision year (e.g. 2016) — the paper whose future citations the model has to predict from author history alone.
- **Left side**: a small representative slice of the **pre-$y$** citation graph (year $< 2016$). Real cite edges between the visible papers are drawn in faint gray.
- **Green nodes**: positives — papers actually cited by $u$.
- **Red nodes**: negatives — papers in the same pre-$y$ graph that $u$ does **not** cite (sampled by the prep notebook).
- **Black nodes**: a few other pre-$y$ papers shown only to anchor the graph structure.
- **Green arrows from $u$**: the supervision signal — "these are the edges the model is asked to score above all the rest".

We deliberately do **not** draw arrows from $u$ to the red nodes. Their red color already says "in the graph, not cited"; adding lines for non-edges would just clutter the picture.

We also keep the node count small (≈15 left + 1 right). The pedagogical point is the *role* of each node, not the size of the real graph — that's what `v2_data_analysis.ipynb` is for.

## Imports + plot style

Mirrors `v2_data_analysis.ipynb`: 150 DPI, tight bbox on save, mid-size font, so this figure drops straight into the report alongside the EDA figures.

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

plt.rcParams.update({
    'font.size': 11,
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})

## Configuration

Path detection is defensive so the notebook runs from either project root or `notebooks/`. We use `processed_v2/` for the supervision bundles.

Knobs you might want to touch:

- `TRAIN_YEAR` — the supervision year. The pre-graph is everything `< TRAIN_YEAR`.
- `N_POS_SHOW`, `N_NEG_SHOW`, `N_CTX_SHOW` — node budget per role. Keep small for legibility.
- `SEED` — controls both example selection and `spring_layout` placement, so the figure is reproducible.
- `MIN_POS`, `MAX_POS` — range of `|positives|` an author must have to be picked. Avoids degenerate (1 positive) and overcrowded (>8) examples.


In [ ]:
# --- Path detection (run from project root or from notebooks/) ---
_here = Path.cwd()
_root = next((c for c in [_here, _here.parent] if (c / 'data').exists()), _here)
DATA_DIR = next((_root / 'data' / sub for sub in ['processed_v2', 'processed']
                 if (_root / 'data' / sub).exists()), None)
FIG_DIR = _root / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR is not None, f"Could not find data/processed_v2 or data/processed under {_root}"
print(f"Project root: {_root}")
print(f"Data dir:     {DATA_DIR}")
print(f"Figure dir:   {FIG_DIR}")

# --- Visualization knobs ---
TRAIN_YEAR = 2016
N_POS_SHOW = 5
N_NEG_SHOW = 5
N_CTX_SHOW = 7        # graph-context nodes; bumped up so more existing cite edges show
MIN_POS, MAX_POS = 5, 7
SEED = 7

random.seed(SEED)
np.random.seed(SEED)

## Step 1 — Load one supervision year

Each `train_year_<y>.pt` carries (i) the pre-$y$ heterogeneous graph, (ii) the per-author supervision examples, and (iii) the local-id maps. We only need the `hetero_graph` and `supervision` here.

The supervision example splits negatives into `negatives_random` and `negatives_hard`. The helper `get_negs` flattens both into one list for the figure.


In [ ]:
bundle = torch.load(DATA_DIR / f'train_year_{TRAIN_YEAR}.pt', weights_only=False)
g = bundle['hetero_graph']
supervision = bundle['supervision']

ex0 = supervision[0]
schema = 'v2' if 'negatives_random' in ex0 else 'v1'

def get_negs(ex):
    """Return all negatives for an example, regardless of v1/v2 schema."""
    if 'negative_locals' in ex:
        return list(ex['negative_locals'])
    return list(ex.get('negatives_random', [])) + list(ex.get('negatives_hard', []))

# Cite edges (directed): we only need ('paper', 'cites', 'paper').
# 'cited_by' is the same edges reversed and would double-draw.
cite_ei = g['paper', 'cites', 'paper'].edge_index
all_cites = list(zip(cite_ei[0].tolist(), cite_ei[1].tolist()))
n_papers = g['paper'].num_nodes

print(f"Schema: {schema}")
print(f"Pre-{TRAIN_YEAR} graph: {n_papers:,} papers, {len(all_cites):,} directed cite edges")
print(f"Supervision examples: {len(supervision):,}")
print(f"Example keys: {list(ex0.keys())}")

## Step 2 — Pick a clean example

We want an example where the figure is intelligible at a glance:

- `MIN_POS ≤ |positives| ≤ MAX_POS` — enough green nodes to look meaningful, few enough to fit.
- enough negatives to choose from after sampling.
- non-empty author history (the model can't even compute an author embedding otherwise).

If multiple authors qualify we shuffle and pick the first, so different `SEED` values give a different but equally clean figure.

In [ ]:
candidates = [
    (i, ex) for i, ex in enumerate(supervision)
    if MIN_POS <= len(ex['positive_locals']) <= MAX_POS
    and len(get_negs(ex)) >= N_NEG_SHOW + 2
    and 1 <= len(ex['history_locals'])
]
print(f"Eligible authors: {len(candidates):,}")

random.shuffle(candidates)
ex_idx, example = candidates[0]

print(f"\nPicked example #{ex_idx}")
print(f"  author_id (local): {example['author_id']}")
print(f"  positives: {len(example['positive_locals'])}")
print(f"  negatives: {len(get_negs(example))}")
print(f"  history:   {len(example['history_locals'])}  paper(s)")

## Step 3 — Sample the visible nodes

From the picked example, sample `N_POS_SHOW` positives and `N_NEG_SHOW` negatives. For the **black context nodes** we don't pick at random — we pick papers that are actually cite-connected to a visible positive or negative, sorted by how many visible papers they touch. This makes the gray cite edges in the final figure non-trivial: there's real graph structure behind the colors, not isolated dots.

Context candidates exclude the author's own history (which is conceptually the query side, not the candidate side).

In [ ]:
from collections import defaultdict

all_pos = list(example['positive_locals'])
all_neg = list(get_negs(example))
random.shuffle(all_pos)
random.shuffle(all_neg)
pos_show = all_pos[:N_POS_SHOW]
neg_show = all_neg[:N_NEG_SHOW]

core_set = set(pos_show) | set(neg_show)
history_set = set(example['history_locals'])

# Treat the cite graph as undirected for *picking* context (direction is shown later via arrowheads).
adj = defaultdict(set)
for s, t in all_cites:
    adj[s].add(t)
    adj[t].add(s)

# Pool of plausible context nodes: papers that touch at least one core node, excluding core+history.
ctx_pool = {n for c in core_set for n in adj[c] if n not in core_set and n not in history_set}

# Greedy: at each step add the candidate that creates the most NEW visible cite edges.
# 'visible' grows as we add — a candidate that connects to an already-added context node still scores.
ctx_show = []
visible_now = set(core_set)
remaining = set(ctx_pool)
while len(ctx_show) < N_CTX_SHOW and remaining:
    best, best_gain = None, -1
    for cand in remaining:
        gain = len(adj[cand] & visible_now)
        if gain > best_gain:
            best, best_gain = cand, gain
    if best is None or best_gain == 0:
        break
    ctx_show.append(best)
    visible_now.add(best)
    remaining.discard(best)

visible_nodes = pos_show + neg_show + ctx_show
visible_set = set(visible_nodes)

# Cite edges only between visible nodes (deduplicated, keep direction for the arrows).
visible_edges = list({(s, t) for s, t in all_cites if s in visible_set and t in visible_set})

print(f"Visible nodes: {len(visible_nodes)}  (pos={len(pos_show)} + neg={len(neg_show)} + ctx={len(ctx_show)})")
print(f"Cite edges between visible nodes: {len(visible_edges)}")

## Step 4 — Layout

Spring layout on the **undirected** version of the visible subgraph (we just want positions, direction is shown via the arrows themselves). We then linearly remap the layout into a fixed left-side rectangle so the picture is balanced and there's room for $u$ on the right.

If the visible subgraph happens to have isolated nodes (no cite edges), spring layout still produces sane positions — they just float a bit further out, which is fine.

In [ ]:
G_layout = nx.Graph()
G_layout.add_nodes_from(visible_nodes)
G_layout.add_edges_from(visible_edges)
raw_pos = nx.spring_layout(G_layout, seed=SEED, k=1.4)

# Remap into a left-side box so u has clear space on the right.
xs = [p[0] for p in raw_pos.values()]
ys = [p[1] for p in raw_pos.values()]
x_min, x_max = min(xs), max(xs)
y_min, y_max = min(ys), max(ys)
X_LO, X_HI = -2.5, 1.0
Y_LO, Y_HI = -2.2, 2.2

def remap(p):
    nx_, ny_ = p
    rx = X_LO + (nx_ - x_min) / max(x_max - x_min, 1e-9) * (X_HI - X_LO)
    ry = Y_LO + (ny_ - y_min) / max(y_max - y_min, 1e-9) * (Y_HI - Y_LO)
    return (rx, ry)

node_pos = {n: remap(p) for n, p in raw_pos.items()}
u_pos = (4.3, 0.0)
print(f"Layout: {len(node_pos)} graph nodes in [{X_LO},{X_HI}] x [{Y_LO},{Y_HI}], u at {u_pos}")

## Step 5 — Render

Drawing order matters: gray internal cite edges first (so they sit behind everything), then green supervision arrows from $u$ (mid-layer), then nodes (top), then $u$ itself. The legend is placed inside the axes so the saved figure is self-contained.

In [ ]:
COLOR_POS = '#2ca02c'        # green — papers cited by u
COLOR_NEG = '#d62728'        # red   — sampled negatives
COLOR_CTX = '#444444'        # near-black — graph context
COLOR_U   = '#1f6fb6'        # blue  — paper u
COLOR_GRAPH_EDGE = '#bdbdbd' # light gray — existing cite edges in pre-y graph

fig, ax = plt.subplots(figsize=(13, 7))

# (1) Existing cite edges between visible pre-y papers — gray, with subtle arrowheads.
for s, t in visible_edges:
    ax.annotate(
        '', xy=node_pos[t], xytext=node_pos[s],
        arrowprops=dict(arrowstyle='-|>', color=COLOR_GRAPH_EDGE,
                        lw=0.9, alpha=0.7, shrinkA=10, shrinkB=10,
                        mutation_scale=10),
        zorder=1,
    )

# (2) Green supervision arrows: u --> each positive.
for p in pos_show:
    ax.annotate(
        '', xy=node_pos[p], xytext=u_pos,
        arrowprops=dict(arrowstyle='-|>', color=COLOR_POS,
                        lw=1.6, alpha=0.9, shrinkA=18, shrinkB=12,
                        mutation_scale=14),
        zorder=2,
    )

# (3) Pre-y graph nodes by role.
def role_color(node_id):
    if node_id in set(pos_show):
        return COLOR_POS
    if node_id in set(neg_show):
        return COLOR_NEG
    return COLOR_CTX

for n in visible_nodes:
    x, y = node_pos[n]
    ax.scatter([x], [y], s=420, c=role_color(n),
               edgecolors='white', linewidths=1.6, zorder=3)

# (4) Paper u — distinctive square marker, larger.
ax.scatter([u_pos[0]], [u_pos[1]], s=900, c=COLOR_U, marker='s',
           edgecolors='white', linewidths=2.0, zorder=4)
ax.text(u_pos[0], u_pos[1] + 0.5, f'paper $u$  (year {TRAIN_YEAR})',
        ha='center', va='bottom', fontsize=12.5, fontweight='bold',
        color=COLOR_U)

# (5) Soft section label for the left half.
ax.text((X_LO + X_HI) / 2, Y_HI + 0.45,
        f'pre-{TRAIN_YEAR} citation graph',
        ha='center', va='bottom', fontsize=12, color='#555555', style='italic')

# (6) Legend, inside the axes, lower-right corner.
legend_handles = [
    mlines.Line2D([], [], color=COLOR_U, marker='s', linestyle='', markersize=12,
                  label=fr'$u$ — paper from {TRAIN_YEAR} (query)'),
    mlines.Line2D([], [], color=COLOR_POS, marker='o', linestyle='', markersize=11,
                  label='positive — cited by $u$'),
    mlines.Line2D([], [], color=COLOR_NEG, marker='o', linestyle='', markersize=11,
                  label='negative — sampled, not cited by $u$'),
    mlines.Line2D([], [], color=COLOR_CTX, marker='o', linestyle='', markersize=11,
                  label=f'other pre-{TRAIN_YEAR} paper (graph context)'),
    mlines.Line2D([], [], color=COLOR_POS, lw=2,
                  label=r'$u \rightarrow$ cited paper (training positive)'),
    mlines.Line2D([], [], color=COLOR_GRAPH_EDGE, lw=1.4,
                  label='existing cite edge in pre-$y$ graph'),
]
ax.legend(handles=legend_handles, loc='lower right',
          fontsize=10, frameon=True, framealpha=0.95)

ax.set_title(
    f'One training example: paper $u$ from {TRAIN_YEAR} and its citation candidates',
    fontsize=13.5, pad=12,
)
ax.set_xlim(-3.4, 6.2)
ax.set_ylim(Y_LO - 0.6, Y_HI + 1.0)
ax.set_aspect('equal')
ax.axis('off')

out_path = FIG_DIR / 'training_example.png'
plt.savefig(out_path)
plt.show()
print(f'\nSaved {out_path}')

## Notes

- The author's history papers are intentionally **not** drawn. Conceptually they belong to the query side ($u$) — in the model, $u$'s representation is a mean-pool of those embeddings. Putting them in the same picture as the candidates would conflate "input" with "label space". For a separate diagram of the author-history side, draw history nodes near $u$ with dotted lines.
- We don't draw lines from $u$ to red nodes. Their red color already carries the "sampled, not cited" meaning; adding non-edges would just clutter the picture.
- Re-run with a different `SEED` (or different `TRAIN_YEAR` / `MIN_POS`) to pick another author. Layout is reproducible per seed.